# FCM-LSTM on Close Price
Fuzzy C-Means memberships (fit on train only) are appended to the scaled close price, then an LSTM predicts the next-day close. One `.keras` model is saved per ticker.

In [1]:
import os, json, random
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

XLSX = "14c9c5b1-EBT_-_Update2.xlsx"
SHEETS = ["DGIK","JARR","TGRA","PTBA","POWR","PGEO","MPOW","ARKO","APEX"]
WINDOW, N_CLUSTERS, FUZZ_M = 20, 3, 2.0
TRAIN_FRAC, VAL_FRAC = 0.8, 0.1
EPOCHS, BATCH = 100, 32
OUT = "models"; os.makedirs(OUT, exist_ok=True)

I0000 00:00:1790835822.485293    5866 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790835822.531415    5866 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790835824.126625    5866 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
class FCM:
    """Fuzzy C-Means. fit() on train data, predict() gives memberships for any data."""
    def __init__(self, c=3, m=2.0, max_iter=200, tol=1e-6, seed=42):
        self.c, self.m, self.max_iter, self.tol, self.seed = c, m, max_iter, tol, seed

    def _u(self, X):
        d = np.linalg.norm(X[:, None, :] - self.centers[None], axis=2) + 1e-12
        inv = d ** (-2.0 / (self.m - 1))
        return inv / inv.sum(axis=1, keepdims=True)

    def fit(self, X):
        rng = np.random.default_rng(self.seed)
        U = rng.random((len(X), self.c)); U /= U.sum(axis=1, keepdims=True)
        for _ in range(self.max_iter):
            Um = U ** self.m
            self.centers = (Um.T @ X) / Um.sum(axis=0)[:, None]
            U_new = self._u(X)
            if np.abs(U_new - U).max() < self.tol: U = U_new; break
            U = U_new
        order = np.argsort(self.centers[:, 0])      # sort clusters low -> high price
        self.centers = self.centers[order]
        return self

    def predict(self, X):
        return self._u(X)

In [3]:
def load_close(sheet):
    df = pd.read_excel(XLSX, sheet_name=sheet, usecols=["Date","Close"])
    df["Date"] = pd.to_datetime(df["Date"])
    return df.sort_values("Date").reset_index(drop=True)

def make_windows(F, y, window):
    X, Y = [], []
    for i in range(window, len(F)):
        X.append(F[i-window:i]); Y.append(y[i])
    return np.array(X, dtype="float32"), np.array(Y, dtype="float32")

def build_model(n_features):
    m = keras.Sequential([
        layers.Input((WINDOW, n_features)),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return m

def metrics(y, p):
    return dict(RMSE=float(np.sqrt(np.mean((y-p)**2))), MAE=float(np.mean(np.abs(y-p))),
                MAPE=float(np.mean(np.abs((y-p)/y))*100))

In [4]:
def run(sheet):
    df = load_close(sheet); close = df["Close"].values.astype("float64")
    n = len(close); n_tr, n_va = int(n*TRAIN_FRAC), int(n*VAL_FRAC)

    # scaling and FCM are fit on the train segment only (no leakage)
    lo, hi = close[:n_tr].min(), close[:n_tr].max()
    sc = (close - lo) / (hi - lo)
    ret = np.r_[0, np.diff(close)/close[:-1]]
    r_sd = ret[:n_tr].std() + 1e-12
    fcm_in = np.c_[sc, np.clip(ret/(4*r_sd), -1, 1)]
    fcm = FCM(N_CLUSTERS, FUZZ_M).fit(fcm_in[:n_tr])
    U = fcm.predict(fcm_in)

    F = np.c_[sc, U]                                  # features: close + memberships
    delta = np.r_[0, np.diff(sc)]; d_sd = delta[:n_tr].std() + 1e-12   # target: next-day change in scaled close
    X, y = make_windows(F, delta/d_sd, WINDOW)
    t = np.arange(WINDOW, n)                           # target index of each window
    tr, va, te = t < n_tr, (t >= n_tr) & (t < n_tr+n_va), t >= n_tr+n_va

    model = build_model(F.shape[1])
    cb = [keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5)]
    model.fit(X[tr], y[tr], validation_data=(X[va], y[va]), epochs=EPOCHS,
              batch_size=BATCH, shuffle=False, callbacks=cb, verbose=0)

    naive = close[t[te]-1]                              # yesterday's close baseline
    pred = naive + model.predict(X[te], verbose=0).ravel()*d_sd*(hi-lo)   # close_t = close_{t-1} + predicted change
    true = close[t[te]]
    path = f"{OUT}/FCM_LSTM_{sheet}.keras"; model.save(path)
    np.savez(f"{OUT}/FCM_LSTM_{sheet}_params.npz", lo=lo, hi=hi, r_sd=r_sd, d_sd=d_sd, centers=fcm.centers)

    r = dict(Ticker=sheet, N=n, Test=int(te.sum()), **metrics(true, pred),
             Naive_RMSE=metrics(true, naive)["RMSE"], Naive_MAPE=metrics(true, naive)["MAPE"])
    return r, df["Date"].values[t[te]], true, pred

results, preds = [], {}
for s in SHEETS:
    r, d, tr_, pr_ = run(s); results.append(r); preds[s] = (d, tr_, pr_); print(r)
res = pd.DataFrame(results).round(3); res.to_csv("fcm_lstm_results.csv", index=False); res

{'Ticker': 'DGIK', 'N': 1373, 'Test': 138, 'RMSE': 2.0242610941572563, 'MAE': 1.3919939774616132, 'MAPE': 1.7126408401383062, 'Naive_RMSE': 2.0305635685191197, 'Naive_MAPE': 1.6782929340715333}


{'Ticker': 'JARR', 'N': 743, 'Test': 75, 'RMSE': 73.0247039785139, 'MAE': 36.63598364278372, 'MAPE': 4.250051581062577, 'Naive_RMSE': 69.12751502356593, 'Naive_MAPE': 3.98845627200645}


{'Ticker': 'TGRA', 'N': 1373, 'Test': 138, 'RMSE': 1.030109478015123, 'MAE': 0.5477800959777991, 'MAPE': 1.5267332042801753, 'Naive_RMSE': 1.028577179648058, 'Naive_MAPE': 1.445414548524151}


{'Ticker': 'PTBA', 'N': 1373, 'Test': 138, 'RMSE': 54.40685533604494, 'MAE': 29.92205351245487, 'MAPE': 1.1563064089920823, 'Naive_RMSE': 54.493717228606315, 'Naive_MAPE': 1.1472460803103812}


{'Ticker': 'POWR', 'N': 1373, 'Test': 138, 'RMSE': 6.919695416264884, 'MAE': 4.626895163356556, 'MAPE': 0.6744614191268687, 'Naive_RMSE': 6.92872617273995, 'Naive_MAPE': 0.6500366760112638}


{'Ticker': 'PGEO', 'N': 599, 'Test': 61, 'RMSE': 52.942672083476815, 'MAE': 40.243033842187465, 'MAPE': 2.6225181278994927, 'Naive_RMSE': 52.95760754340505, 'Naive_MAPE': 2.6229622549848943}


{'Ticker': 'MPOW', 'N': 1373, 'Test': 138, 'RMSE': 5.629290684969403, 'MAE': 3.3654499902206467, 'MAPE': 2.9107507964493182, 'Naive_RMSE': 5.636963927402816, 'Naive_MAPE': 2.901531822645955}


{'Ticker': 'ARKO', 'N': 762, 'Test': 77, 'RMSE': 47.5775009969487, 'MAE': 29.582761606306203, 'MAPE': 2.6966290317215322, 'Naive_RMSE': 46.77939384292713, 'Naive_MAPE': 2.669648577452511}


{'Ticker': 'APEX', 'N': 1373, 'Test': 138, 'RMSE': 6.057957072327295, 'MAE': 3.33273723541141, 'MAPE': 2.8289144496710965, 'Naive_RMSE': 6.0672559038579, 'Naive_MAPE': 2.8303726189343004}


,Ticker,N,Test,RMSE,MAE,MAPE,Naive_RMSE,Naive_MAPE
0,DGIK,1373,138,2.024,1.392,1.713,2.031,1.678
1,JARR,743,75,73.025,36.636,4.250,69.128,3.988
2,TGRA,1373,138,1.030,0.548,1.527,1.029,1.445
3,PTBA,1373,138,54.407,29.922,1.156,54.494,1.147
4,POWR,1373,138,6.920,4.627,0.674,6.929,0.650
5,PGEO,599,61,52.943,40.243,2.623,52.958,2.623
6,MPOW,1373,138,5.629,3.365,2.911,5.637,2.902
7,ARKO,762,77,47.578,29.583,2.697,46.779,2.670
8,APEX,1373,138,6.058,3.333,2.829,6.067,2.830


In [5]:
# sanity check: reload a saved .keras and predict
m = keras.models.load_model(f"{OUT}/FCM_LSTM_PTBA.keras"); m.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                   │ (None, 20, 64)         │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 91,877 (358.90 KB)

 Trainable params: 30,625 (119.63 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 61,252 (239.27 KB)

In [6]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(3, 3, figsize=(15, 9))
for a, s in zip(ax.ravel(), SHEETS):
    d, t_, p_ = preds[s]; a.plot(d, t_, label="Actual"); a.plot(d, p_, label="FCM-LSTM"); a.set_title(s)
ax[0,0].legend(); plt.tight_layout(); plt.savefig("fcm_lstm_test_predictions.png", dpi=110)